# Project 3: Calgary Traffic Incident Hotspot Analyzer
## Exploratory Data Analysis

This notebook performs exploratory data analysis on Calgary's traffic incident data, sourced from the City of Calgary Open Data portal (dataset ID: `35ra-9556`).

**Objectives:**
- Load and inspect the raw data
- Examine incident distribution by quadrant, hour, and day of week
- Visualize incident locations on a scatter map
- Analyze temporal trends over months and years

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to path so we can import src modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_and_prepare_data, create_clustering_features

# Plotly default template
TEMPLATE = "plotly_white"
COLOR_SEQ = ["#4A90D9", "#7B68EE", "#6C5CE7", "#00B894", "#FDCB6E", "#E17055"]

print("Imports complete.")

## 1. Data Loading and Basic Statistics

In [ ]:
# Load and preprocess data
df = load_and_prepare_data(limit=100000)

print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {len(df):,}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nColumn names:\n{list(df.columns)}")

In [ ]:
# Display basic info and first few rows
df.info()

In [ ]:
df.head(10)

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_count", ascending=False)

print("Missing values:")
missing_report[missing_report["missing_count"] > 0]

## 2. Incident Count by Quadrant

In [ ]:
if "quadrant" in df.columns:
    quad_counts = df["quadrant"].value_counts().reset_index()
    quad_counts.columns = ["quadrant", "count"]

    fig = px.bar(
        quad_counts,
        x="quadrant",
        y="count",
        color="quadrant",
        color_discrete_sequence=COLOR_SEQ,
        template=TEMPLATE,
        title="Traffic Incidents by Quadrant",
        labels={"count": "Number of Incidents", "quadrant": "Quadrant"},
    )
    fig.update_layout(showlegend=False, height=450)
    fig.show()

    print(f"\nQuadrant counts:\n{quad_counts.to_string(index=False)}")
else:
    print("Quadrant column not found.")

## 3. Incident Count by Hour of Day

In [ ]:
if "hour" in df.columns:
    hourly = df.groupby("hour").size().reset_index(name="count")

    fig = px.bar(
        hourly,
        x="hour",
        y="count",
        color="count",
        color_continuous_scale=["#7B68EE", "#4A90D9", "#D63031"],
        template=TEMPLATE,
        title="Traffic Incidents by Hour of Day",
        labels={"hour": "Hour (0-23)", "count": "Number of Incidents"},
    )
    fig.update_layout(height=450)
    fig.show()

    peak_hour = hourly.loc[hourly["count"].idxmax(), "hour"]
    print(f"\nPeak hour: {int(peak_hour):02d}:00")
    print(f"Lowest hour: {int(hourly.loc[hourly['count'].idxmin(), 'hour']):02d}:00")
else:
    print("Hour column not found.")

## 4. Incident Count by Day of Week

In [ ]:
if "day_of_week" in df.columns:
    day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    if "day_name" in df.columns:
        daily = df["day_name"].value_counts().reindex(day_order).reset_index()
        daily.columns = ["day", "count"]
    else:
        daily = df.groupby("day_of_week").size().reset_index(name="count")
        daily["day"] = daily["day_of_week"].map({i: d for i, d in enumerate(day_order)})

    fig = px.bar(
        daily,
        x="day",
        y="count",
        color="count",
        color_continuous_scale=["#6C5CE7", "#4A90D9"],
        template=TEMPLATE,
        title="Traffic Incidents by Day of Week",
        labels={"day": "Day", "count": "Number of Incidents"},
    )
    fig.update_layout(height=450)
    fig.show()

    weekend_count = daily[daily["day"].isin(["Saturday", "Sunday"])]["count"].sum()
    weekday_count = daily[~daily["day"].isin(["Saturday", "Sunday"])]["count"].sum()
    print(f"\nWeekday total: {weekday_count:,}")
    print(f"Weekend total: {weekend_count:,}")
    print(f"Ratio (weekday avg / weekend avg): {(weekday_count / 5) / (weekend_count / 2):.2f}")
else:
    print("Day of week column not found.")

## 5. Scatter Plot of Incidents on Map

In [ ]:
# Sample for performance (plotting all points can be slow)
sample_size = min(10000, len(df))
df_sample = df.sample(n=sample_size, random_state=42)

fig = px.scatter_mapbox(
    df_sample,
    lat="latitude",
    lon="longitude",
    color="quadrant" if "quadrant" in df_sample.columns else None,
    color_discrete_sequence=COLOR_SEQ,
    zoom=10,
    center={"lat": 51.05, "lon": -114.07},
    mapbox_style="carto-positron",
    opacity=0.5,
    title=f"Traffic Incident Locations (sample of {sample_size:,})",
    hover_data={"latitude": ":.4f", "longitude": ":.4f"},
)
fig.update_layout(height=600, margin={"l": 0, "r": 0, "t": 40, "b": 0})
fig.show()

print(f"Latitude range: [{df['latitude'].min():.4f}, {df['latitude'].max():.4f}]")
print(f"Longitude range: [{df['longitude'].min():.4f}, {df['longitude'].max():.4f}]")

## 6. Temporal Trend Analysis

In [ ]:
# Monthly trend
if "month" in df.columns:
    month_names = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    monthly = df.groupby("month").size().reset_index(name="count")
    monthly["month_name"] = monthly["month"].apply(
        lambda m: month_names[int(m)] if 1 <= int(m) <= 12 else str(m)
    )

    fig = px.bar(
        monthly,
        x="month_name",
        y="count",
        color="count",
        color_continuous_scale=["#4A90D9", "#7B68EE"],
        template=TEMPLATE,
        title="Traffic Incidents by Month",
        labels={"month_name": "Month", "count": "Number of Incidents"},
    )
    fig.update_layout(height=450)
    fig.show()

In [ ]:
# Year-over-year trend
if "year" in df.columns:
    yearly = df.groupby("year").size().reset_index(name="count")
    yearly = yearly[yearly["year"] > 2000]  # Filter unreasonable years

    fig = px.line(
        yearly,
        x="year",
        y="count",
        markers=True,
        template=TEMPLATE,
        title="Traffic Incidents - Year-over-Year Trend",
        labels={"year": "Year", "count": "Number of Incidents"},
        color_discrete_sequence=["#4A90D9"],
    )
    fig.update_layout(height=450)
    fig.show()

    print(f"\nYear-over-year counts:\n{yearly.to_string(index=False)}")

In [ ]:
# Heatmap: hour vs day of week
if "hour" in df.columns and "day_of_week" in df.columns:
    heatmap_data = (
        df.groupby(["day_of_week", "hour"])
        .size()
        .reset_index(name="count")
        .pivot(index="day_of_week", columns="hour", values="count")
        .fillna(0)
    )
    day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    heatmap_data.index = [
        day_labels[i] if i < len(day_labels) else str(i)
        for i in heatmap_data.index
    ]

    fig = px.imshow(
        heatmap_data,
        aspect="auto",
        color_continuous_scale=["#DFE6E9", "#7B68EE", "#D63031"],
        template=TEMPLATE,
        title="Incident Heatmap: Hour of Day vs Day of Week",
        labels={"x": "Hour of Day", "y": "Day of Week", "color": "Incidents"},
    )
    fig.update_layout(height=450)
    fig.show()

## 7. Summary

**Key observations from EDA:**

1. **Quadrant distribution**: Incidents are not evenly distributed across Calgary's four quadrants, with some quadrants seeing significantly higher counts.
2. **Hourly pattern**: Clear rush-hour peaks in the morning (7-9 AM) and afternoon (3-6 PM), with a lull overnight.
3. **Day of week**: Weekdays dominate incident counts; weekends are notably quieter.
4. **Spatial clusters**: Visual inspection of the scatter map reveals dense clusters along major corridors (Deerfoot Trail, Crowchild Trail, Stoney Trail).
5. **Seasonal/monthly variation**: Some months show higher incident counts, potentially correlating with weather or road conditions.

These patterns motivate the use of spatial clustering (DBSCAN, KMeans) and temporal classification (Random Forest, Gradient Boosting) in the modeling phase.